# Part 3 — Make your own analysis!

For each project, the objective is to implement your own likelihood analysis.

* Implement:
   * Implement the signal and background models
   * Perform the fit of signal parameter(s) (number of signal events for most of the exercises and another one for Project VII)
   * Prepare the code to compute p-value, following what was shown in Part 2
   * __Optional:__ Prepare also the code for upper limit and/or confidence interval
* Estimate:
   * Compute $2\sigma$ (95%)  discovery potential
   * __Optional:__ Compute sensitivity at 90% CL
* Unblind!
   * We provide you a generated dataset to be unblinded! 
   * Compute the best-fit of the signal parameters
   * Compute the p-value (quantifying the rejection of the background hypothesis)
   * __Optional:__ compute confidence intervals or upper limits

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from scipy.optimize import minimize_scalar
from tqdm import tqdm

# Settings: font size for plot labels
matplotlib.rcParams['font.size'] = 14

In [2]:
# Useful functions (maybe to adapt)

def generate_component(mu, pdf, fluctuate=True, **kwargs):
    n = np.random.poisson(lam=mu) if fluctuate else int(mu)
    return pdf.rvs(size=n, **kwargs)

def generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf, fluctuate_signal=True, **kwargs):
    signal = generate_component(mu_signal, signal_pdf, fluctuate=fluctuate_signal, **kwargs)
    background = generate_component(mu_background, background_pdf)
    return np.concatenate((signal, background)), len(signal), len(background)

def loglikelihood(data, mu_signal, mu_background, signal_pdf, background_pdf):
    l1 = mu_signal * signal_pdf.pdf(data) + mu_background * background_pdf.pdf(data)
    return np.log(l1).sum() - (mu_signal + mu_background)

def likelihood_fit(data, mu_background, signal_pdf, background_pdf):
    if len(data)==0: return 0,0
    l1 = minimize_scalar(lambda mu: -loglikelihood(data, mu, mu_background, signal_pdf, background_pdf), bounds=(0, len(data))) 
    return l1.x, -l1.fun

def compute_Lambda(data, mu_background, signal_pdf, background_pdf):
    if len(data) == 0: return 0
    loglike_H1 = likelihood_fit(data, mu_background, signal_pdf, background_pdf)[1]
    return 2*(loglike_H1 - loglikelihood(data, 0, mu_background, signal_pdf, background_pdf))

def get_Lambda_distribution(mu_signal, mu_background, signal_pdf, background_pdf, Npseudo_experiments = 10000, fluctuate_signal=True):
    Lambda_table = np.zeros(Npseudo_experiments)
    for i in tqdm(range(Npseudo_experiments)):
        data, _, _ = generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf, fluctuate_signal=fluctuate_signal)
        Lambda_table[i] = compute_Lambda(data, mu_background, signal_pdf, background_pdf)
    return Lambda_table

def compute_lambdamu(data, mu, mu_background, signal_pdf, background_pdf):
    loglike_H1 = likelihood_fit(data, mu_background, signal_pdf, background_pdf)[1]
    ts = 2*(loglike_H1 - loglikelihood(data, mu, mu_background, signal_pdf, background_pdf))
    return ts

def get_lambdamu_distribution(mu, mu_signal, mu_background, signal_pdf, background_pdf, Npseudo_experiments = 10000, fluctuate_signal=True):
    Lambda_table = np.zeros(Npseudo_experiments)
    for i in tqdm(range(Npseudo_experiments)):
        data, _, _ = generate_dataset(mu_signal, mu_background, signal_pdf, background_pdf, fluctuate_signal=fluctuate_signal)
        Lambda_table[i] = compute_lambdamu(data, mu, mu_background, signal_pdf, background_pdf)
    return Lambda_table

# For the signal and background PDFs, you may use scipy.stats functions or your own class following this template:
# Instead of these functions, you could also design your own classes defining the same methods
class example_pdf:
    def __init__(self, mu, sigma):
        self.mu, self.sigma = mu, sigma

    def pdf(self, x, sigma=None):
        return norm.pdf(x, loc=self.mu, scale=sigma if sigma is not None else self.sigma)

    def rvs(self, size, sigma=None):
        return norm.rvs(loc=self.mu, scale=sigma if sigma is not None else self.sigma, size=size)

### Project I — A triangular case (difficulty: 1/5)

Define a triangular signal function from -5 to 5, peaking at 0. For the background, we use a uniform distribution from -20 to 20. The expected total number of background events mu_b is 100.

__What you will need to implement/modify/run:__
- define your own signal and background PDF
- run existing codes for discovery/p-value/sensitivity

### Project II — Energy spectrum fit (difficulty: 1/5)

This exercise is analogous to a spectral analysis. The signal and the background PDFs are two power laws of the form $(\gamma+1) \left(\dfrac{E}{\text{1 TeV}}\right)^{​−\gamma}$, with $\gamma=2$ and 4.2, respectively. These power laws are set to 0 below 1 TeV. Such truncated power laws can be described using the `pareto` function of `scipy.stats`, with shape parameter $b=\gamma−1$.

__What you will need to implement/modify/run:__
- define your own signal and background PDF
- run existing codes for discovery/p-value/sensitivity

### Project III — 2D in 1D (difficulty: 2/5)

We are going to get closer to investigating a point-like source. We will assume that the signal and background pdfs depend only on the localization and define the signal pdf as
$$S(x,y) = e^{-\frac{x^2 + y^2}{2}}.$$
The background is taken to be uniformly distributed over the 2D space up to a given distance from the source. The expected number of background events is mu_b = 300.

1. We would like to reduce this 2D problem to a 1D problem. Which observable can we collapse x and y into? What would the signal and background PDFs look like?


2. The signal PDF as a function of the 1D observable can be described by a Rayleigh distribution or by a Weibull distribution with $c=2$. A truncated form of the latter, `truncweibull_min`, is a `scipy.stats` function. Use it to describe the signal PDF, truncated at a given maximal distance to the source.


3. Use scipy.stats function to parameterize the background PDF. Bonus: use instead your own sampling method.


Follow the rest of the instructions at the beginning of the notebook, putting the cutoff at 10.

__What you will need to implement/modify/run:__
- define your own signal and background PDF
- run existing codes for discovery/p-value/sensitivity

### Project IV — A new dimension (difficulty: 3/5)

We add an additional dimension to the problem, with both signal and background being characterised by two parameters, x and y. For background, we use a 2D Gaussian distribution centered on $(0, 0)$ with $\sigma_x = \sigma_y = 1$ and no correlations. For signal, we use a 2D Gaussian distribution centered on $(2, 0)$ with $\sigma_x = \sigma_y = 1$ and 50% correlation between the two parameters. The expected number of background events is $\mu_b = 50$.

__What you will need to implement/modify/run:__
- define your own signal and background PDF
- run existing codes for discovery/p-value/sensitivity... with a minor adaptation of `generate_component`

### Project V — An angular problem (difficulty: 4/5)

We consider a problem where the data have as characteristics a direction in spherical coordinates $(\theta, \phi)$. The background is uniformly distributed in $\phi$ and in $\cos\theta$. The signal follows a Von Mises - Fisher distribution $S(\theta, \phi) = \text{VonMisesFisher}(\theta, \phi, \theta_\text{src}, \phi_\text{src}, \kappa = 1/\sigma^2)$ with $\sigma=5\degree$.

We take $\theta_\text{src} = 12.5\degree$, $\phi_\text{src} = 42.4\degree$, and $\mu_b = 500$ is the expected number of background events.

__What you will need to implement/modify/run:__
- define your own signal and background PDF using classes to accomodate the usage of $\theta$ and $\phi$
- run existing codes for discovery/p-value/sensitivity... with a minor adaptation of `generate_component`

### Project VI — Binned likelihood (difficulty: 4/5)

So far we have seen how to implement an **unbinned** likelihood fit. When the statistics is large, it can be more computationally efficient to bin the data and fit histograms (**binned** likelihood). In this exercise, you will implement your own binned likelihood algorithm and use it to perform the fit presented in class (Gaussian signal + uniform background).

1. From a Gaussian PDF ($\sigma=1$, $x\in [-10,10]$), make a histogram with bin width $0.2$ (hint: use the trapezoid method to go faster). For a total number of events of $N_\mathrm{tot} = 1000$, evaluate the expected number of events in each bin. Do the same for the background PDF (uniform in $[-10,10]$). **Store these values in arrays**.


2. If $\lambda_i$ events are expected in bin $i$, what is the probability distribution of the observed number of events $n_i$ in that bin? Consider that each bin is independent from the other and hence that this number can freely fluctuate.
  

3. How can you quickly generate a set of $n_i$ values in Python? Rewrite the `generate_component` function to generate a random histogram instead of a random list of observables. Then, rewrite `generate_dataset`. With this method, can you still force the total number of signal events to be fixed?


4. Use answer 2 to write a general formula for the binned log-likelihood for $N_\mathrm{bin}$ bins as a function of the $\lambda_i$.
  
  
5. Which computationally expensive term can you drop from the likelihood without affecting the test-statistic? Use the answer and the formula computed above to modify `loglikelihood`.


6. From your answers to question 3, do you now need to generate pseudo-experiments for non-integer $\mu_s$ values? Which part(s) of the existing codes do you still need to modify?


7. Use your new code to redo the Gaussian + uniform fit shown in class.

  
7. **Bonus**: We can write $\lambda_i = \mu_s s_i + \mu_b b_i$ with $\mu_s$ and $\mu_b$ the signal and background strengths, and $s_i,b_i$ the values of the signal and background pdfs in bin $i$. Assuming $\mu_b$ is fixed, write a simplified formula for the test-statistic (denoting the best-fit $\mu_s$ as $\hat\mu_s$).
  
  
8. **Bonus**: how could you modify the previous code to generate pseudo-experiments only for integer $\mu_s$ values? (hint: check out `scipy.stats.multinomial`)
 

9. **Bonus**: find the advantages and drawbacks of the binned likelihood approach.


**Advice**: in this exercise, some functions from `part2_unbinned_likelihood.ipynb` will need to be thoroughly modified. When you are modifying a function, try to keep the number of return values constant (e.g. replacing a return value by `None` instead of removing it), and keep the same function arguments even if some are no longer used. This approach will allow you to directly reuse the other parts of the code.

**WARNING!!!** If you want to work on another exercise after this one do not forget to restart the kernel to go back to the former definitions!


In [ ]:
binwidth = 10
xmin = -10
xmax = 10
sigma = 1

In [ ]:
# PDF values for all bins
sample_background_pdf = ...
sample_signal_pdf = ...

In [38]:
def generate_component(mu, pdf_h):
    ...
    return n

def generate_dataset(mu_signal, mu_background, signal_pdf_h, background_pdf_h, fluctuate_signal=False):
    signal = ...
    background = ...
    return signal + background, None, None

def loglikelihood(data, mu_signal, mu_background, signal_pdf_h, background_pdf_h):
    logl1 = ...
    return logl1.sum()

### Project VII — Why signal? Why? (difficulty: 5/5)

We consider both signal and background to be characterised by two parameters, x and y. For background, we use a 2D Gaussian distribution centered on $(0, 0)$ with $\sigma_x = \sigma_y = 1$ and no correlations. For signal, we use a 2D Gaussian distribution centered on $(2, y_0)$ with $\sigma_x = \sigma_y = 1$ and 50% correlation between the two parameters. The expected number of background events is $\mu_b = 50$ and $y_0$ is **unknown** between -10 and +10.

__What you will need to implement/modify/run:__
- define your own signal and background PDF
- adapt likelihood fit and Lambda computation to accomodate for the additional parameter
- think about how to compute and show discovery potentials, sensitivities...

In [60]:
from scipy.optimize import minimize

def loglikelihood(data, mu_signal, mu_background, signal_pdf, background_pdf, **kwargs):
    ...
    return ...

def likelihood_fit(data, mu_background, signal_pdf, background_pdf):
    l1 = ...
    return ...

def compute_Lambda(data, mu_background, signal_pdf, background_pdf):
    return ...

def get_Lambda_distribution(mu_signal, mean_y_signal, mu_background, signal_pdf, background_pdf, Npseudo_experiments=10000, fluctuate_signal=True):
    Lambda_table = np.zeros(Npseudo_experiments)
    ...
    return Lambda_table